# Optimization Space: How Training Selects a Hypothesis


## 0. Learning objectives and lesson structure

The previous notebooks separated evidence and possibility:

$$
\mathcal{D}\quad\text{is the observed data,}
\qquad
\mathcal{H}\quad\text{is the set of candidate functions.}
$$

This notebook isolates the optimisation ingredient:

$$
\boxed{\mathcal{H}} + \boxed{\mathcal{D}} + \boxed{\mathcal{O}} \rightarrow s.
$$

The central question is:

$$
\text{Given data }\mathcal{D}\text{ and candidates }\mathcal{H},\text{ how does training select one solution }s?
$$

A common answer is an optimisation rule that searches for parameters with small empirical loss:

$$
\widehat{\theta}
\approx
\arg\min_{\theta\in\Theta}
\widehat{R}_{\mathcal{D}}(h_{\theta}),
\qquad
\widehat{h}=h_{\widehat{\theta}}.
$$

By the end, students should be able to:

- define an objective function from data, hypotheses, loss, and regularisation;
- distinguish empirical risk from population risk;
- explain gradient descent as an iterative search procedure;
- describe how learning rate and curvature affect optimisation behaviour;
- explain why stochastic gradients introduce variance into the training path and can shape the distribution over selected solutions;
- interpret regularisation, early stopping, and optimiser choices as selection pressures;
- connect optimisation paths in parameter space to selected functions in hypothesis space.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import Markdown, display
except Exception:  # pragma: no cover - only used outside notebooks
    Markdown = None
    display = print

SEED = 7
rng = np.random.default_rng(SEED)

COLORS = {
    "data": "#2F5D7C",
    "truth": "#222222",
    "selected": "#C7502A",
    "alt": "#7B5E9E",
    "regularised": "#2F7D4A",
    "sgd": "#D18F24",
    "support": "#D9D9D9",
    "validation": "#5B8CC0",
}

LINE_GRID = np.linspace(-3.2, 3.2, 400)
RESIDUAL_GRID = np.linspace(-4.0, 4.0, 400)
VIRIDIS_CMAP = "viridis"

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "legend.frameon": False,
})


def set_seed(seed=SEED):
    return np.random.default_rng(seed)


def display_markdown(text):
    if Markdown is None:
        print(text)
    else:
        display(Markdown(text))


def format_cell(value):
    if isinstance(value, np.ndarray):
        return "[" + ", ".join(f"{v:.3f}" for v in value) + "]"
    if isinstance(value, (float, np.floating)):
        return f"{value:.4g}"
    text = str(value)
    return text.replace("|", "\\|")


def markdown_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separator = "|" + "|".join(["---"] * len(headers)) + "|"
    body = ["| " + " | ".join(format_cell(cell) for cell in row) + " |" for row in rows]
    return "\n".join([header, separator, *body])


def display_table(headers, rows):
    display_markdown(markdown_table(headers, rows))


def make_axis(title, xlabel="x", ylabel="y", figsize=(7, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return fig, ax


def true_linear_function(x):
    return 0.35 + 0.8 * np.asarray(x)


def make_linear_dataset(n=28, seed=SEED, outlier=True):
    rng = set_seed(seed)
    x = np.sort(rng.uniform(-2.6, 2.6, n))
    y = true_linear_function(x) + rng.normal(0.0, 0.22, n)
    if outlier:
        x = np.concatenate([x, [2.35]])
        y = np.concatenate([y, [-2.15]])
    return x, y


def linear_design(x):
    x = np.asarray(x)
    return np.column_stack([np.ones_like(x), x])


def linear_predictions(X, theta):
    return X @ theta


def squared_loss(pred, y):
    return (pred - y) ** 2


def absolute_loss(pred, y):
    return np.abs(pred - y)


def huber_loss(residual, delta=1.0):
    residual = np.asarray(residual)
    abs_r = np.abs(residual)
    return np.where(abs_r <= delta, 0.5 * residual**2, delta * (abs_r - 0.5 * delta))


def mse_objective(X, y, theta, lam=0.0):
    residual = X @ theta - y
    return np.mean(residual**2) + lam * np.sum(theta**2)


def mae_objective(X, y, theta):
    return np.mean(np.abs(X @ theta - y))


def mse_gradient(X, y, theta, lam=0.0):
    n = X.shape[0]
    return (2 / n) * X.T @ (X @ theta - y) + 2 * lam * theta


def ridge_solution(X, y, lam=0.0):
    n, p = X.shape
    gram = X.T @ X + n * lam * np.eye(p)
    rhs = X.T @ y
    try:
        return np.linalg.solve(gram, rhs)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(gram) @ rhs


def fit_linear_absolute_grid(x, y, intercept_range=(-1.8, 1.8), slope_range=(-0.3, 1.8), points=241):
    intercepts = np.linspace(*intercept_range, points)
    slopes = np.linspace(*slope_range, points)
    best_loss = np.inf
    best_theta = None
    for intercept in intercepts:
        pred = intercept + slopes[:, None] * x[None, :]
        losses = np.mean(np.abs(pred - y[None, :]), axis=1)
        idx = int(np.argmin(losses))
        if losses[idx] < best_loss:
            best_loss = float(losses[idx])
            best_theta = np.array([intercept, slopes[idx]])
    return best_theta, best_loss


def gradient_descent(theta0, grad_fn, obj_fn, eta, steps):
    theta = np.array(theta0, dtype=float)
    path = [theta.copy()]
    values = [float(obj_fn(theta))]
    for _ in range(steps):
        theta = theta - eta * grad_fn(theta)
        path.append(theta.copy())
        values.append(float(obj_fn(theta)))
    return np.asarray(path), np.asarray(values)


def sgd_path(X, y, theta0, eta, steps, batch_size, seed=SEED, lam=0.0):
    rng = set_seed(seed)
    theta = np.array(theta0, dtype=float)
    path = [theta.copy()]
    values = [mse_objective(X, y, theta, lam=lam)]
    n = X.shape[0]
    replace = batch_size > n
    for _ in range(steps):
        idx = rng.choice(n, size=batch_size, replace=replace)
        theta = theta - eta * mse_gradient(X[idx], y[idx], theta, lam=lam)
        path.append(theta.copy())
        values.append(mse_objective(X, y, theta, lam=lam))
    return np.asarray(path), np.asarray(values)


def plot_observed(ax, x, y, label="observed data", color=None, alpha=0.88):
    ax.scatter(x, y, s=35, color=color or COLORS["data"], edgecolor="white", linewidth=0.4, alpha=alpha, label=label)


def plot_linear_fit(ax, theta, x_grid=LINE_GRID, label=None, color=None, lw=2.5, ls="-"):
    X_grid = linear_design(x_grid)
    ax.plot(x_grid, X_grid @ theta, color=color or COLORS["selected"], lw=lw, ls=ls, label=label)


def objective_grid_linear(x, y, intercepts, slopes, loss="mse", lam=0.0):
    I, S = np.meshgrid(intercepts, slopes, indexing="ij")
    residual = I[..., None] + S[..., None] * x[None, None, :] - y[None, None, :]
    if loss == "mae":
        Z = np.mean(np.abs(residual), axis=2)
    else:
        Z = np.mean(residual**2, axis=2)
    if lam:
        Z = Z + lam * (I**2 + S**2)
    return I, S, Z


def status_from_eta(eta, a):
    factor = abs(1 - eta * a)
    if eta <= 0:
        return "invalid"
    if factor < 0.25:
        return "fast stable"
    if factor < 1 and (1 - eta * a) < 0:
        return "oscillating stable"
    if factor < 1:
        return "slow stable"
    return "divergent"


def describe_lambda(lam):
    if lam == 0:
        return "least constrained fit"
    if lam < 1e-3:
        return "weak norm preference"
    if lam < 0.1:
        return "smoother compatible fit"
    return "strong norm preference"


def make_loss_mesh(theta0_range, theta1_range, points=120):
    theta0_values = np.linspace(theta0_range[0], theta0_range[1], points)
    theta1_values = np.linspace(theta1_range[0], theta1_range[1], points)
    return np.meshgrid(theta0_values, theta1_values, indexing="ij")


def evaluate_objective_mesh(T0, T1, objective_fn):
    flat = np.column_stack([T0.ravel(), T1.ravel()])
    values = np.array([objective_fn(theta) for theta in flat])
    return values.reshape(T0.shape)


def plot_3d_landscape(ax, T0, T1, Z, title, path=None, objective_fn=None, selected=None):
    ax.plot_surface(T0, T1, Z, cmap=VIRIDIS_CMAP, alpha=0.88, linewidth=0, antialiased=True)
    if path is not None and objective_fn is not None:
        path = np.asarray(path)
        z_path = np.array([objective_fn(theta) for theta in path])
        ax.plot(path[:, 0], path[:, 1], z_path, color=COLORS["selected"], lw=2.4, marker="o", ms=3.0, label="optimisation path")
    if selected is not None and objective_fn is not None:
        selected = np.asarray(selected)
        ax.scatter(selected[0], selected[1], objective_fn(selected), color=COLORS["selected"], s=55, label=r"$\theta_T$")
    ax.set_title(title)
    ax.set_xlabel(r"$\theta_0$")
    ax.set_ylabel(r"$\theta_1$")
    ax.set_zlabel(r"$J(\theta)$")
    ax.view_init(elev=28, azim=-132)
    ax.legend(fontsize=7)

print("Setup complete: deterministic helpers, losses, optimisers, and plotting utilities are ready.")


## 1. From a hypothesis space to one selected solution

A hypothesis space contains many possible functions:

$$
\mathcal{H}=\{h_{\theta}:\theta\in\Theta\}.
$$

Optimisation supplies a rule for selecting one of them. For supervised data

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n},
$$

an empirical objective often has the form

$$
J(\theta)
=
\widehat{R}_{\mathcal{D}}(h_{\theta})+\lambda\Omega(\theta),
$$

where

$$
\widehat{R}_{\mathcal{D}}(h_{\theta})
=
\frac{1}{n}\sum_{i=1}^{n}\ell(h_{\theta}(x_i),y_i)
$$

measures fit to observed data, and $\Omega$ encodes an additional preference or penalty.

When training is finite, the reported solution is the realised function at the returned iterate:

$$
\theta_0,\theta_1,\ldots,\theta_T,
\qquad
s=h_{\theta_T}.
$$

This makes $\mathcal{O}$ more than a loss function. It includes the optimiser, initialisation, learning-rate schedule, stopping rule, stochasticity, and any regularisation.

| Component of $\mathcal{O}$ | What changed? | What function was selected? |
|---|---|---|
| Loss | squared, absolute, cross-entropy, Huber | Different residuals or decisions dominate. |
| Regularisation | $\lambda$, penalty form | Smaller norm, smoother, or otherwise preferred solution. |
| Learning rate | $\eta$, schedule | Different path stability and possible final point. |
| Batch size | full batch, minibatch | Different path variance and solution distribution. |
| Initialisation | $\theta_0$ | Different basin or parameter solution. |
| Stopping | early, validation-selected, late | Different degree of fit/noise-fitting. |

> **Researcher note.** The trained model is not determined by $\mathcal{D}$ and $\mathcal{H}$ alone. It is determined by the full selection procedure $\mathcal{O}$.


In [ ]:
x_demo, y_demo = make_linear_dataset(seed=11, outlier=True)
X_demo = linear_design(x_demo)

rules = []
theta_squared = ridge_solution(X_demo, y_demo, lam=0.0)
rules.append(("squared loss", "loss", theta_squared, mse_objective(X_demo, y_demo, theta_squared), "large residuals dominate"))

theta_absolute, abs_loss_value = fit_linear_absolute_grid(x_demo, y_demo)
rules.append(("absolute loss", "loss", theta_absolute, abs_loss_value, "outlier influence is reduced"))

theta_ridge = ridge_solution(X_demo, y_demo, lam=0.45)
rules.append(("squared + ridge", "regularisation", theta_ridge, mse_objective(X_demo, y_demo, theta_ridge, lam=0.45), "smaller-norm line preferred"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_observed(axes[0], x_demo, y_demo)
for label, _, theta, _, _ in rules:
    color = {"squared loss": COLORS["selected"], "absolute loss": COLORS["alt"], "squared + ridge": COLORS["regularised"]}[label]
    ls = "--" if label == "absolute loss" else "-"
    plot_linear_fit(axes[0], theta, label=label, color=color, ls=ls)
axes[0].set_title("Selected two-parameter linear functions")
axes[0].set_xlim(-3.1, 3.1)
axes[0].set_ylim(-3.0, 3.2)
axes[0].legend(loc="upper left")

intercepts = np.linspace(-1.0, 1.1, 120)
slopes = np.linspace(0.0, 1.4, 120)
I, S, Z = objective_grid_linear(x_demo, y_demo, intercepts, slopes, loss="mse")
axes[1].contourf(I, S, Z, levels=24, cmap=VIRIDIS_CMAP)
for label, _, theta, _, _ in rules:
    marker = "*" if label == "squared loss" else "o"
    axes[1].scatter(theta[0], theta[1], s=70, marker=marker, label=label)
axes[1].set_title("2D loss landscape over two parameters")
axes[1].set_xlabel(r"$\theta_0$")
axes[1].set_ylabel(r"$\theta_1$")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(14, 4.3))
for j, (title, loss, lam, theta) in enumerate([
    ("squared loss surface", "mse", 0.0, theta_squared),
    ("absolute loss surface", "mae", 0.0, theta_absolute),
    ("ridge surface", "mse", 0.45, theta_ridge),
], start=1):
    ax = fig.add_subplot(1, 3, j, projection="3d")
    _, _, Z_rule = objective_grid_linear(x_demo, y_demo, intercepts, slopes, loss=loss, lam=lam)
    objective = (lambda th, loss=loss, lam=lam: mae_objective(X_demo, y_demo, th) if loss == "mae" else mse_objective(X_demo, y_demo, th, lam=lam))
    plot_3d_landscape(ax, I, S, Z_rule, title, selected=theta, objective_fn=objective)
plt.tight_layout()
plt.show()

display_table(
    ["Rule", "Ingredient changed", "Training loss under active loss", "Selected parameters", "Interpretation"],
    [(label, ingredient, loss_value, theta, interpretation) for label, ingredient, theta, loss_value, interpretation in rules],
)


> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution?

| Ingredient | Held fixed? | Changed? | Consequence |
|---|---:|---:|---|
| $\mathcal{D}$ | yes | no | The observed evidence is identical in all three fits. |
| $\mathcal{H}$ | yes | no | Every selected function is still a straight line. |
| $\mathcal{O}$ | no | yes | The objective selects a different line. |
| Estimand | no | yes | Squared and absolute loss target different summaries of residual behaviour. |
| Deployment distribution | yes | no | We have not changed where the fitted line will be used. |


## 2. Loss functions define what counts as a mistake

The loss function maps a prediction and target to a penalty:

$$
\ell(\widehat{y},y)\geq 0.
$$

For regression, squared loss and absolute loss are

$$
\ell_2(\widehat{y},y)=(\widehat{y}-y)^2,
\qquad
\ell_1(\widehat{y},y)=|\widehat{y}-y|.
$$

With residual $r=\widehat{y}-y$, squared loss emphasises large residuals because its derivative grows with residual size:

$$
\ell_2(r)=r^2,
\qquad
\frac{d}{dr}r^2=2r.
$$

For binary classification with $y\in\{0,1\}$ and predicted probability $p_{\theta}(x)$, cross-entropy loss is

$$
\ell_{\mathrm{CE}}(p,y)
=-y\log p-(1-y)\log(1-p).
$$

For squared-error prediction, the population minimiser is a conditional mean. For absolute-error prediction, the population minimiser is a conditional median. A loss function therefore encodes what kind of summary or decision the model is trained to make.


In [ ]:
x_clean, y_clean = make_linear_dataset(seed=11, outlier=False)
x_outlier, y_outlier = make_linear_dataset(seed=11, outlier=True)

def fit_squared_and_absolute(x, y):
    X = linear_design(x)
    theta_sq = ridge_solution(X, y, lam=0.0)
    theta_abs, _ = fit_linear_absolute_grid(x, y)
    return theta_sq, theta_abs

theta_clean_sq, theta_clean_abs = fit_squared_and_absolute(x_clean, y_clean)
theta_out_sq, theta_out_abs = fit_squared_and_absolute(x_outlier, y_outlier)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
residual = RESIDUAL_GRID
axes[0].plot(residual, residual**2, color=COLORS["selected"], lw=2.5, label="squared")
axes[0].plot(residual, np.abs(residual), color=COLORS["alt"], lw=2.5, label="absolute")
axes[0].plot(residual, huber_loss(residual, delta=1.0), color=COLORS["regularised"], lw=2.5, label="Huber, delta=1")
axes[0].set_title("Residual penalties")
axes[0].set_xlabel("residual r")
axes[0].set_ylabel("loss")
axes[0].set_ylim(0, 8)
axes[0].legend()

plot_observed(axes[1], x_clean, y_clean)
plot_linear_fit(axes[1], theta_clean_sq, label="squared", color=COLORS["selected"])
plot_linear_fit(axes[1], theta_clean_abs, label="absolute", color=COLORS["alt"], ls="--")
axes[1].set_title("Without outlier")
axes[1].set_xlim(-3.1, 3.1)
axes[1].set_ylim(-3.0, 3.2)
axes[1].legend()

plot_observed(axes[2], x_outlier, y_outlier)
plot_linear_fit(axes[2], theta_out_sq, label="squared", color=COLORS["selected"])
plot_linear_fit(axes[2], theta_out_abs, label="absolute", color=COLORS["alt"], ls="--")
axes[2].set_title("With one outlier")
axes[2].set_xlim(-3.1, 3.1)
axes[2].set_ylim(-3.0, 3.2)
axes[2].legend()
plt.tight_layout()
plt.show()

intercepts = np.linspace(-1.0, 1.1, 120)
slopes = np.linspace(0.0, 1.4, 120)
I, S, Z_sq = objective_grid_linear(x_outlier, y_outlier, intercepts, slopes, loss="mse")
_, _, Z_abs = objective_grid_linear(x_outlier, y_outlier, intercepts, slopes, loss="mae")
X_outlier = linear_design(x_outlier)

fig = plt.figure(figsize=(11.5, 4.5))
for j, (title, Z, theta, objective) in enumerate([
    ("Squared loss surface with outlier", Z_sq, theta_out_sq, lambda th: mse_objective(X_outlier, y_outlier, th)),
    ("Absolute loss surface with outlier", Z_abs, theta_out_abs, lambda th: mae_objective(X_outlier, y_outlier, th)),
], start=1):
    ax = fig.add_subplot(1, 2, j, projection="3d")
    plot_3d_landscape(ax, I, S, Z, title, selected=theta, objective_fn=objective)
plt.tight_layout()
plt.show()

rows = []
for name, x, y, theta_sq, theta_abs in [
    ("without outlier", x_clean, y_clean, theta_clean_sq, theta_clean_abs),
    ("with outlier", x_outlier, y_outlier, theta_out_sq, theta_out_abs),
]:
    X = linear_design(x)
    rows.append((name, "squared", mse_objective(X, y, theta_sq), mae_objective(X, y, theta_sq), theta_sq))
    rows.append((name, "absolute", mse_objective(X, y, theta_abs), mae_objective(X, y, theta_abs), theta_abs))

display_table(["Dataset", "Selected by", "MSE", "MAE", "Selected parameters"], rows)


> **Researcher note.** Changing the loss can change the scientific target. A line selected by squared loss supports a different residual-summary claim than a line selected by absolute loss.

> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution?


## 3. Objective landscapes in parameter space

For a linear model

$$
h_{\theta}(x)=\theta_0+\theta_1x,
$$

with squared loss, the empirical objective is

$$
J(\theta_0,\theta_1)
=
\frac{1}{n}\sum_{i=1}^{n}(\theta_0+\theta_1x_i-y_i)^2.
$$

In matrix form, with design matrix $X$, target vector $y$, and parameter vector $\theta$,

$$
J(\theta)=\frac{1}{n}\|X\theta-y\|_2^2.
$$

This objective is a convex quadratic. Its gradient is

$$
\nabla J(\theta)=\frac{2}{n}X^{\top}(X\theta-y),
$$

and the Hessian is

$$
\nabla^2J(\theta)=\frac{2}{n}X^{\top}X.
$$

Feature scaling can change the optimisation landscape even when the statistical prediction problem is unchanged. This matters because finite-time optimisation follows the geometry it is given.


In [ ]:
rng = set_seed(19)
x_wide = np.linspace(0, 90, 45)
y_wide = 1.2 + 0.045 * x_wide + rng.normal(0, 0.55, len(x_wide))
X_unscaled = linear_design(x_wide)
x_scaled = (x_wide - x_wide.mean()) / x_wide.std()
X_scaled = linear_design(x_scaled)

theta_unscaled = ridge_solution(X_unscaled, y_wide)
theta_scaled = ridge_solution(X_scaled, y_wide)
cond_unscaled = np.linalg.cond(X_unscaled.T @ X_unscaled)
cond_scaled = np.linalg.cond(X_scaled.T @ X_scaled)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
surface_payloads = []
configs = [
    (axes[0, 0], axes[0, 1], "unscaled x", x_wide, X_unscaled, theta_unscaled, cond_unscaled),
    (axes[1, 0], axes[1, 1], "scaled x", x_scaled, X_scaled, theta_scaled, cond_scaled),
]

for contour_ax, fit_ax, title, x_feature, X, theta_hat, cond in configs:
    intercepts = np.linspace(theta_hat[0] - 2.3, theta_hat[0] + 2.3, 150)
    slope_width = 0.08 if title == "unscaled x" else 2.0
    slopes = np.linspace(theta_hat[1] - slope_width, theta_hat[1] + slope_width, 150)
    I, S, Z = objective_grid_linear(x_feature, y_wide, intercepts, slopes)
    surface_payloads.append((title, I, S, Z, theta_hat, lambda th, X=X: mse_objective(X, y_wide, th)))
    levels = np.linspace(float(np.min(Z)), float(np.quantile(Z, 0.88)), 16)
    contour_ax.contourf(I, S, Z, levels=levels, cmap=VIRIDIS_CMAP)
    contour_ax.scatter(theta_hat[0], theta_hat[1], color=COLORS["selected"], s=55, label="least-squares solution")
    offsets = [np.array([0.8, 0.0]), np.array([0.0, 0.45 * slope_width]), np.array([-0.8, -0.35 * slope_width])]
    for j, offset in enumerate(offsets, start=1):
        candidate = theta_hat + offset
        contour_ax.scatter(candidate[0], candidate[1], color=COLORS["alt"], alpha=0.75, s=30)
        if title == "unscaled x":
            fit_ax.plot(x_wide, linear_design(x_wide) @ candidate, color=COLORS["alt"], alpha=0.45, lw=1.5, label="nearby parameter" if j == 1 else None)
        else:
            fit_ax.plot(x_wide, linear_design((x_wide - x_wide.mean()) / x_wide.std()) @ candidate, color=COLORS["alt"], alpha=0.45, lw=1.5, label="nearby parameter" if j == 1 else None)
    contour_ax.set_title(f"{title}: condition number {cond:.1f}")
    contour_ax.set_xlabel(r"$\theta_0$")
    contour_ax.set_ylabel(r"$\theta_1$")
    contour_ax.legend()

    plot_observed(fit_ax, x_wide, y_wide)
    if title == "unscaled x":
        fit_ax.plot(x_wide, X_unscaled @ theta_hat, color=COLORS["selected"], lw=2.6, label="selected function")
    else:
        fit_ax.plot(x_wide, X_scaled @ theta_hat, color=COLORS["selected"], lw=2.6, label="selected function")
    fit_ax.set_title(f"Functions represented by contour points ({title})")
    fit_ax.set_xlabel("original x")
    fit_ax.set_ylabel("y")
    fit_ax.legend()

plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(12.5, 4.5))
for j, (title, I, S, Z, theta_hat, objective) in enumerate(surface_payloads, start=1):
    ax = fig.add_subplot(1, 2, j, projection="3d")
    plot_3d_landscape(ax, I, S, Z, f"3D {title} loss landscape", selected=theta_hat, objective_fn=objective)
plt.tight_layout()
plt.show()

display_table(
    ["Representation", "condition number of X^T X", "selected parameters", "Consequence"],
    [
        ("unscaled x", cond_unscaled, theta_unscaled, "elongated landscape"),
        ("scaled x", cond_scaled, theta_scaled, "rounder landscape"),
    ],
)


> **Researcher note.** Feature scaling is not only cosmetic. With finite optimisation budgets, the coordinate geometry can decide whether training reaches the function that the objective would eventually prefer.

> **Checkpoint.** What can the learner see? The same prediction problem can produce very different parameter-space contours, so optimiser movement is shaped by the representation of $\mathcal{H}$.


## 4. Gradient descent as iterative search

Gradient descent updates parameters by moving against the objective gradient:

$$
\theta_{t+1}=\theta_t-\eta\nabla J(\theta_t),
$$

where $\eta>0$ is the learning rate. In this notebook the landscape is two-dimensional because the model has two parameters:

$$
h_{\theta}(x)=\theta_0+\theta_1x.
$$

For squared-loss linear regression,

$$
J(\theta)=\frac{1}{n}\|X\theta-y\|_2^2,
\qquad
\nabla J(\theta)=\frac{2}{n}X^\top(X\theta-y).
$$

The same update rule can move slowly, converge quickly, oscillate, or diverge depending on the learning rate and the curvature of this two-parameter loss surface.

> **Researcher note.** Learning rate is not only a runtime parameter. When training is finite, unstable, or path-dependent, it can affect which solution is selected.


In [ ]:
x_gd, y_gd = make_linear_dataset(n=36, seed=13, outlier=False)
x_gd_scaled = (x_gd - x_gd.mean()) / x_gd.std()
X_gd = linear_design(x_gd_scaled)
theta_star = ridge_solution(X_gd, y_gd)
hessian = (2 / len(y_gd)) * X_gd.T @ X_gd
eta_limit = 2 / np.linalg.eigvalsh(hessian).max()
theta0 = np.array([-1.1, 1.6])
steps = 18
eta_presets = [0.12 * eta_limit, 0.62 * eta_limit, 0.94 * eta_limit, 1.08 * eta_limit]
labels = ["slow stable", "fast stable", "oscillating stable", "divergent"]

objective = lambda th: mse_objective(X_gd, y_gd, th)
gradient = lambda th: mse_gradient(X_gd, y_gd, th)
T0, T1 = make_loss_mesh((theta_star[0] - 1.8, theta_star[0] + 1.8), (theta_star[1] - 1.8, theta_star[1] + 1.8), points=140)
Z = evaluate_objective_mesh(T0, T1, objective)

fig = plt.figure(figsize=(15, 4.8))
ax_surface = fig.add_subplot(1, 3, 1, projection="3d")
fast_path, _ = gradient_descent(theta0, gradient, objective, eta_presets[1], steps)
plot_3d_landscape(ax_surface, T0, T1, Z, "3D two-parameter loss surface", path=fast_path, objective_fn=objective, selected=fast_path[-1])

ax_contour = fig.add_subplot(1, 3, 2)
levels = np.linspace(float(np.min(Z)), float(np.quantile(Z, 0.92)), 22)
ax_contour.contourf(T0, T1, Z, levels=levels, cmap=VIRIDIS_CMAP)
rows = []
for eta, label in zip(eta_presets, labels):
    path, values = gradient_descent(theta0, gradient, objective, eta, steps)
    color = COLORS["regularised"] if label == "slow stable" else COLORS["selected"] if label == "fast stable" else COLORS["alt"] if label == "oscillating stable" else COLORS["sgd"]
    ax_contour.plot(path[:, 0], path[:, 1], marker="o", ms=3.0, lw=1.5, color=color, label=label)
    rows.append((label, eta, eta_limit, path[-1], values[-1]))
ax_contour.scatter(theta_star[0], theta_star[1], color=COLORS["truth"], s=65, marker="*", label=r"$\widehat\theta$")
ax_contour.set_title("Learning-rate paths on the same landscape")
ax_contour.set_xlabel(r"$\theta_0$")
ax_contour.set_ylabel(r"$\theta_1$")
ax_contour.legend(fontsize=7)

ax_function = fig.add_subplot(1, 3, 3)
plot_observed(ax_function, x_gd_scaled, y_gd, label="training data")
z_grid = np.linspace(x_gd_scaled.min() - 0.25, x_gd_scaled.max() + 0.25, 220)
for eta, label in zip(eta_presets[:3], labels[:3]):
    path, _ = gradient_descent(theta0, gradient, objective, eta, steps)
    color = COLORS["regularised"] if label == "slow stable" else COLORS["selected"] if label == "fast stable" else COLORS["alt"]
    plot_linear_fit(ax_function, path[-1], x_grid=z_grid, label=label, color=color)
plot_linear_fit(ax_function, theta_star, x_grid=z_grid, label="least-squares solution", color=COLORS["truth"], ls=":")
ax_function.set_title("Finite-time selected functions")
ax_function.set_xlabel("scaled x")
ax_function.set_ylabel("y")
ax_function.legend(fontsize=7)
plt.tight_layout()
plt.show()

display_table(["Preset", "eta", "2/lambda_max", "theta_T", "J(theta_T)"], rows)


> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? Here only $\mathcal{O}$ changed: the learning rate changed the path and, for finite $T$, the selected iterate.


## 5. Learning rate, curvature, and conditioning

For a multivariate quadratic objective

$$
J(\theta)=\frac{1}{2}(\theta-\theta^{\star})^{\top}A(\theta-\theta^{\star}),
$$

where $A$ is symmetric positive definite, the gradient is

$$
\nabla J(\theta)=A(\theta-\theta^{\star}).
$$

Gradient descent updates errors according to

$$
\theta_{t+1}-\theta^{\star}
=(I-\eta A)(\theta_t-\theta^{\star}).
$$

Convergence depends on the eigenvalues of $A$:

$$
0<\eta<\frac{2}{\lambda_{\max}(A)}.
$$

The condition number

$$
\kappa(A)=\frac{\lambda_{\max}(A)}{\lambda_{\min}(A)}
$$

controls how stretched the landscape is. Poor conditioning does not merely make optimisation slow. With finite training budgets, it can leave some directions under-optimised, which changes the selected function.


In [ ]:
theta_star_2d = np.array([0.4, -0.8])
theta0_2d = np.array([2.8, 2.4])
eta = 0.22
steps = 28
A_well = np.diag([2.0, 4.0])
A_ill = np.diag([0.15, 8.0])

def quadratic_path(A, theta0, theta_star, eta, steps):
    def obj(theta):
        d = theta - theta_star
        return 0.5 * d @ A @ d
    def grad(theta):
        return A @ (theta - theta_star)
    return gradient_descent(theta0, grad, obj, eta, steps), obj

path_payloads = {
    "well-conditioned": quadratic_path(A_well, theta0_2d, theta_star_2d, eta, steps),
    "ill-conditioned": quadratic_path(A_ill, theta0_2d, theta_star_2d, eta, steps),
}

fig = plt.figure(figsize=(16, 9))
t0 = np.linspace(-0.4, 3.1, 150)
t1 = np.linspace(-1.4, 2.7, 150)
T0, T1 = np.meshgrid(t0, t1)
for idx, (label, A) in enumerate([("well-conditioned", A_well), ("ill-conditioned", A_ill)], start=1):
    D0 = T0 - theta_star_2d[0]
    D1 = T1 - theta_star_2d[1]
    Z = 0.5 * (A[0, 0] * D0**2 + A[1, 1] * D1**2)
    (path, values), obj = path_payloads[label]

    ax_surface = fig.add_subplot(2, 3, idx, projection="3d")
    plot_3d_landscape(ax_surface, T0, T1, Z, f"3D {label} surface", path=path, objective_fn=obj, selected=path[-1])

    ax_contour = fig.add_subplot(2, 3, idx + 2)
    levels = np.linspace(float(np.min(Z)), float(np.quantile(Z, 0.88)), 18)
    ax_contour.contourf(T0, T1, Z, levels=levels, cmap=VIRIDIS_CMAP)
    ax_contour.plot(path[:, 0], path[:, 1], marker="o", ms=3.2, lw=1.6, color=COLORS["selected"], label="GD path")
    ax_contour.scatter(theta_star_2d[0], theta_star_2d[1], color=COLORS["truth"], s=55, marker="*", label=r"$\theta^\star$")
    ax_contour.scatter(path[-1, 0], path[-1, 1], color=COLORS["sgd"], s=48, label=r"$\theta_T$")
    ax_contour.set_title(f"{label}, kappa={np.linalg.cond(A):.1f}")
    ax_contour.set_xlabel(r"$\theta_0$")
    ax_contour.set_ylabel(r"$\theta_1$")
    ax_contour.legend(fontsize=8)

ax_function = fig.add_subplot(2, 3, 6)
z_grid = np.linspace(-2.5, 2.5, 200)
ax_function.plot(z_grid, theta_star_2d[0] + theta_star_2d[1] * z_grid, color=COLORS["truth"], lw=2.5, label=r"target $h_{\theta^\star}$")
for label, color in [("well-conditioned", COLORS["selected"]), ("ill-conditioned", COLORS["alt"] )]:
    theta_T = path_payloads[label][0][0][-1]
    ax_function.plot(z_grid, theta_T[0] + theta_T[1] * z_grid, color=color, lw=2.4, label=f"{label} theta_T")
ax_function.set_title("Finite-time selected two-parameter functions")
ax_function.set_xlabel("input z")
ax_function.set_ylabel(r"$h_\theta(z)$")
ax_function.legend(fontsize=8)
plt.tight_layout()
plt.show()

rows = []
for label, A in [("well-conditioned", A_well), ("ill-conditioned", A_ill)]:
    path, values = path_payloads[label][0]
    rows.append((label, np.linalg.cond(A), eta, path[-1], values[-1], "closer to target" if label == "well-conditioned" else "shallow direction remains under-trained"))

display_table(["Landscape", "kappa(A)", "eta", "theta_T", "J(theta_T)", "Function consequence"], rows)


> **Researcher note.** A model can be statistically capable of representing a good function while finite optimisation still selects a poorer one because the path moves slowly along important directions.

> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? The demo changes optimiser geometry and finite-time dynamics inside $\mathcal{O}$.


## 6. Stochastic optimisation and minibatches

Full-batch gradient descent uses every data point in each update:

$$
\nabla J(\theta)
=
\frac{1}{n}\sum_{i=1}^{n}\nabla_{\theta}\ell(h_{\theta}(x_i),y_i).
$$

Minibatch stochastic gradient descent uses a subset $B_t\subset\{1,\ldots,n\}$:

$$
g_t
=
\frac{1}{|B_t|}\sum_{i\in B_t}\nabla_{\theta}\ell(h_{\theta}(x_i),y_i),
$$

and updates

$$
\theta_{t+1}=\theta_t-\eta g_t.
$$

Under uniform minibatch sampling, $g_t$ is an unbiased estimator of the full gradient:

$$
\mathbb{E}[g_t\mid\theta_t]=\nabla J(\theta_t).
$$

However, the stochastic training path has variance,

$$
\operatorname{Var}(g_t\mid\theta_t)>0,
$$

and the distribution over final selected solutions can be shaped by that stochasticity. The estimator can be unbiased while the final trained solution is still influenced by the stochastic path.


In [ ]:
x_sgd, y_sgd = make_linear_dataset(n=80, seed=41, outlier=False)
X_sgd = linear_design(x_sgd)
theta_init = np.array([-1.2, 1.7])
eta = 0.035
steps = 90
batch_sizes = [1, 8, 32, len(x_sgd)]
seeds = [3, 4, 5]

runs = []
for batch_size in batch_sizes:
    for seed in (seeds if batch_size < len(x_sgd) else [0]):
        path, values = sgd_path(X_sgd, y_sgd, theta_init, eta=eta, steps=steps, batch_size=batch_size, seed=seed)
        runs.append((batch_size, seed, path, values))

theta_hat = ridge_solution(X_sgd, y_sgd)
objective = lambda th: mse_objective(X_sgd, y_sgd, th)
T0, T1 = make_loss_mesh((theta_hat[0] - 1.8, theta_hat[0] + 1.8), (theta_hat[1] - 1.8, theta_hat[1] + 1.8), points=130)
Z = evaluate_objective_mesh(T0, T1, objective)

fig = plt.figure(figsize=(16, 9))
ax_surface = fig.add_subplot(2, 2, 1, projection="3d")
representative_path = next(path for batch_size, seed, path, values in runs if batch_size == 8 and seed == 3)
plot_3d_landscape(ax_surface, T0, T1, Z, "3D full-data loss surface", path=representative_path, objective_fn=objective, selected=representative_path[-1])

ax_loss = fig.add_subplot(2, 2, 2)
ax_path = fig.add_subplot(2, 2, 3)
ax_function = fig.add_subplot(2, 2, 4)
for batch_size, seed, path, values in runs:
    alpha = 0.95 if batch_size == len(x_sgd) else 0.55
    lw = 2.6 if batch_size == len(x_sgd) else 1.4
    label = "full batch" if batch_size == len(x_sgd) else f"batch={batch_size}, seed={seed}"
    color = COLORS["truth"] if batch_size == len(x_sgd) else COLORS["sgd"] if batch_size == 1 else COLORS["alt"] if batch_size == 8 else COLORS["regularised"]
    ax_loss.plot(values, color=color, alpha=alpha, lw=lw, label=label)
    ax_path.plot(path[:, 0], path[:, 1], color=color, alpha=alpha, lw=lw)

ax_loss.set_title("Objective value over updates")
ax_loss.set_xlabel("update")
ax_loss.set_ylabel("full-data objective")
ax_loss.set_yscale("log")
ax_loss.legend(fontsize=7)

levels = np.linspace(float(np.min(Z)), float(np.quantile(Z, 0.92)), 22)
ax_path.contourf(T0, T1, Z, levels=levels, cmap=VIRIDIS_CMAP, alpha=0.75)
ax_path.scatter(theta_init[0], theta_init[1], color=COLORS["truth"], s=45, marker="x", label=r"$\theta_0$")
ax_path.set_title("Parameter paths on the 2D landscape")
ax_path.set_xlabel(r"$\theta_0$")
ax_path.set_ylabel(r"$\theta_1$")
ax_path.legend(fontsize=8)

plot_observed(ax_function, x_sgd, y_sgd, label="training data", alpha=0.55)
for batch_size, seed, path, values in runs:
    if batch_size in [1, len(x_sgd)] and seed in [3, 0]:
        label = "full batch" if batch_size == len(x_sgd) else "SGD batch=1"
        color = COLORS["truth"] if batch_size == len(x_sgd) else COLORS["sgd"]
        plot_linear_fit(ax_function, path[-1], label=label, color=color)
ax_function.set_title("Selected functions after same update budget")
ax_function.set_xlim(-3.1, 3.1)
ax_function.set_ylim(-3.0, 3.2)
ax_function.legend(fontsize=8)
plt.tight_layout()
plt.show()

rows = []
for batch_size, seed, path, values in runs:
    if batch_size in [1, 8, len(x_sgd)]:
        label = "full" if batch_size == len(x_sgd) else str(batch_size)
        rows.append((label, seed, values[-1], path[-1]))

display_table(["Batch size", "Seed", "Final full-data loss", "theta_T"], rows)


> **Researcher note.** Reporting only the optimiser name can hide an important selection pressure. Batch size, seed, and update budget can change the distribution of selected functions even when the minibatch gradient is unbiased.

> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? The data and linear hypothesis space stayed fixed; stochasticity and batch size changed inside $\mathcal{O}$.


## 7. Regularisation as an explicit selection pressure

Regularisation modifies the objective by adding a penalty:

$$
J_{\lambda}(\theta)
=
\widehat{R}_{\mathcal{D}}(h_{\theta})+\lambda\Omega(\theta),
\qquad \lambda\geq 0.
$$

For the two-parameter linear model $h_\theta(x)=\theta_0+\theta_1x$, ridge regularisation uses

$$
\Omega(\theta)=\|\theta\|_2^2,
$$

so the objective is

$$
J_{\lambda}(\theta)
=
\frac{1}{n}\|X\theta-y\|_2^2+\lambda\|\theta\|_2^2.
$$

For linear least squares, the ridge solution is

$$
\widehat\theta_{\lambda}
=(X^{\top}X+n\lambda I)^{-1}X^{\top}y.
$$

Regularisation is not only a technique for reducing test error. It is a declared preference among functions that may fit the observed data similarly.


In [ ]:
x_reg, y_reg = make_linear_dataset(n=26, seed=31, outlier=True)
X_reg = linear_design(x_reg)
x_val = np.linspace(-2.8, 2.8, 120)
y_val = true_linear_function(x_val)
X_val = linear_design(x_val)
lambdas = [0.0, 0.03, 0.2, 0.8]

theta_unreg = ridge_solution(X_reg, y_reg, lam=0.0)
T0, T1 = make_loss_mesh((theta_unreg[0] - 1.1, theta_unreg[0] + 1.1), (theta_unreg[1] - 1.1, theta_unreg[1] + 1.1), points=130)

fig = plt.figure(figsize=(16, 8.5))
for j, lam in enumerate([0.0, 0.8], start=1):
    objective = lambda th, lam=lam: mse_objective(X_reg, y_reg, th, lam=lam)
    Z = evaluate_objective_mesh(T0, T1, objective)
    theta = ridge_solution(X_reg, y_reg, lam=lam)
    ax = fig.add_subplot(2, 3, j, projection="3d")
    plot_3d_landscape(ax, T0, T1, Z, f"3D ridge surface, lambda={lam:g}", selected=theta, objective_fn=objective)

ax_fit = fig.add_subplot(2, 3, 3)
plot_observed(ax_fit, x_reg, y_reg, label="training data")
rows = []
train_errors = []
val_errors = []
norms = []
for lam in lambdas:
    theta = ridge_solution(X_reg, y_reg, lam=lam)
    train_mse = mse_objective(X_reg, y_reg, theta, lam=0.0)
    val_mse = mse_objective(X_val, y_val, theta, lam=0.0)
    norm = np.linalg.norm(theta)
    train_errors.append(train_mse)
    val_errors.append(val_mse)
    norms.append(norm)
    plot_linear_fit(ax_fit, theta, label=f"lambda={lam:g}", lw=2.1)
    rows.append((lam, train_mse, val_mse, norm, describe_lambda(lam)))
ax_fit.plot(x_val, y_val, color=COLORS["truth"], lw=2.2, ls=":", label="latent reference")
ax_fit.set_title("Same two-parameter H, different lambda")
ax_fit.set_xlim(-3.1, 3.1)
ax_fit.set_ylim(-3.2, 3.1)
ax_fit.legend(fontsize=8)

ax_error = fig.add_subplot(2, 3, 5)
ax_error.plot(lambdas, train_errors, marker="o", label="training MSE", color=COLORS["selected"])
ax_error.plot(lambdas, val_errors, marker="o", label="validation MSE", color=COLORS["validation"])
ax_error.set_title("Error under selected functions")
ax_error.set_xlabel("lambda")
ax_error.set_ylabel("MSE")
ax_error.legend()

ax_norm = fig.add_subplot(2, 3, 6)
ax_norm.plot(lambdas, norms, marker="o", color=COLORS["regularised"])
ax_norm.set_title("Coefficient norm")
ax_norm.set_xlabel("lambda")
ax_norm.set_ylabel(r"$\|\theta\|_2$")
plt.tight_layout()
plt.show()

display_table(["lambda", "Training MSE", "Validation MSE", "norm(theta)", "Selected behaviour"], rows)


> **Researcher note.** When many curves fit the observed points, $\lambda$ states a preference among compatible functions. The claim is not just “less overfitting”; it is “this smoother or smaller-norm solution is the selected one.”

> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? The data and two-parameter hypothesis space stayed fixed; the explicit penalty in $\mathcal{O}$ changed.


## 8. Non-convexity, initialisation, and multiple solutions

Deep-network objectives are generally non-convex:

$$
J(\theta)=\frac{1}{n}\sum_{i=1}^{n}\ell(h_{\theta}(x_i),y_i)
$$

may contain many local minima, saddle points, flat regions, and symmetries. Training starts from an initial parameter vector

$$
\theta_0\sim P_{\mathrm{init}},
$$

then follows an optimiser-defined path. Different initialisations can lead to different parameter solutions:

$$
\theta_T^{(1)}\neq\theta_T^{(2)}.
$$

These parameter solutions may still produce similar functions, or they may generalise differently:

$$
h_{\theta_T^{(1)}}\approx h_{\theta_T^{(2)}}
\quad\text{on training data, but not necessarily elsewhere.}
$$

Optimisation therefore participates in model selection. The selected solution depends on the objective, starting point, optimiser dynamics, stochasticity, and stopping time.


In [ ]:
def nonconvex_objective(theta):
    theta = np.asarray(theta)
    theta0 = theta[..., 0]
    theta1 = theta[..., 1]
    return 0.08 * theta0**2 + 0.06 * theta1**2 + np.sin(3 * theta0) ** 2 + 0.7 * np.sin(3 * theta1) ** 2 + 0.15 * np.sin(2 * theta0 + theta1)


def nonconvex_gradient(theta):
    theta0, theta1 = theta
    d0 = 0.16 * theta0 + 3.0 * np.sin(6 * theta0) + 0.30 * np.cos(2 * theta0 + theta1)
    d1 = 0.12 * theta1 + 2.1 * np.sin(6 * theta1) + 0.15 * np.cos(2 * theta0 + theta1)
    return np.array([d0, d1])

starts = np.array([[-2.8, -2.4], [-2.4, 1.8], [-1.1, -0.4], [-0.2, 2.5], [0.65, -2.2], [1.55, 0.8], [2.65, -0.1], [3.0, 2.4]])
eta = 0.045
steps = 120
T0, T1 = make_loss_mesh((-3.2, 3.2), (-3.0, 3.0), points=170)
Z = nonconvex_objective(np.stack([T0, T1], axis=-1))

fig = plt.figure(figsize=(16, 5.2))
ax_surface = fig.add_subplot(1, 3, 1, projection="3d")
representative_path, _ = gradient_descent(starts[1], nonconvex_gradient, nonconvex_objective, eta, steps)
plot_3d_landscape(ax_surface, T0, T1, Z, "3D non-convex 2D landscape", path=representative_path, objective_fn=nonconvex_objective, selected=representative_path[-1])

ax_contour = fig.add_subplot(1, 3, 2)
levels = np.linspace(float(np.min(Z)), float(np.quantile(Z, 0.9)), 28)
ax_contour.contourf(T0, T1, Z, levels=levels, cmap=VIRIDIS_CMAP)
rows = []
final_thetas = []
for start in starts:
    path, values = gradient_descent(start, nonconvex_gradient, nonconvex_objective, eta, steps)
    ax_contour.plot(path[:, 0], path[:, 1], marker="o", ms=2.1, lw=1.2, alpha=0.78)
    ax_contour.scatter(start[0], start[1], color="white", edgecolor=COLORS["truth"], s=28)
    rows.append((start, path[-1], values[-1]))
    final_thetas.append(path[-1])
ax_contour.set_title("Multiple starts on the same 2D landscape")
ax_contour.set_xlabel(r"$\theta_0$")
ax_contour.set_ylabel(r"$\theta_1$")

ax_functions = fig.add_subplot(1, 3, 3)
z_grid = np.linspace(-2.5, 2.5, 220)
for theta in final_thetas:
    ax_functions.plot(z_grid, theta[0] + theta[1] * z_grid, color=COLORS["alt"], alpha=0.38, lw=1.4)
ax_functions.plot(z_grid, final_thetas[0][0] + final_thetas[0][1] * z_grid, color=COLORS["selected"], lw=2.5, label="one selected function")
ax_functions.set_title("Functions represented by final parameters")
ax_functions.set_xlabel("input z")
ax_functions.set_ylabel(r"$h_\theta(z)$")
ax_functions.legend()
plt.tight_layout()
plt.show()

display_table(["Initial theta_0", "Final theta_T", "Final J(theta_T)"], rows)

display_markdown(
    "This toy objective uses a two-parameter model, so the same run shows both a 2D parameter-space landscape and the functions selected by different final parameters."
)


> **Researcher note.** Different seeds are not a cosmetic detail in a non-convex problem. They define a distribution over selected solutions, so seed sensitivity is evidence about $\mathcal{O}$.

> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? Here the initial point in $\mathcal{O}$ changed, selecting different basins.


## 9. Early stopping and implicit bias

Training returns a finite-time solution:

$$
s=h_{\theta_T},
$$

not necessarily the exact minimiser of the empirical objective. The stopping time $T$ can act like regularisation.

For iterative optimisation,

$$
\theta_0,\theta_1,\ldots,\theta_T,
$$

early iterates may fit broad structure before later iterates fit noise. A validation-based stopping rule is often written as

$$
T^{\star}
=\arg\min_{t\in\{0,\ldots,T_{\max}\}}
\widehat{R}_{\mathrm{val}}(h_{\theta_t}).
$$

Early stopping is a selection rule. It decides which iterate, and therefore which function, becomes the reported solution.


In [ ]:
rng = set_seed(42)
x_train = np.sort(rng.uniform(-2.6, 2.6, 22))
y_train = true_linear_function(x_train) + rng.normal(0.0, 0.16, len(x_train))
x_train = np.concatenate([x_train, [2.45]])
y_train = np.concatenate([y_train, [-5.0]])
X_train = linear_design(x_train)
x_val = np.linspace(-2.8, 2.8, 160)
y_val = true_linear_function(x_val)
X_val = linear_design(x_val)

theta = np.zeros(2)
eta = 0.08
steps = 250
path = []
train_history = []
val_history = []
for _ in range(steps):
    path.append(theta.copy())
    train_history.append(mse_objective(X_train, y_train, theta))
    val_history.append(mse_objective(X_val, y_val, theta))
    theta = theta - eta * mse_gradient(X_train, y_train, theta, lam=0.0)

path = np.asarray(path)
train_history = np.asarray(train_history)
val_history = np.asarray(val_history)
best_t = int(np.argmin(val_history))
checkpoints = {
    "early": 4,
    "validation-selected": best_t,
    "late": steps - 1,
}

objective = lambda th: mse_objective(X_train, y_train, th)
theta_hat = ridge_solution(X_train, y_train)
T0, T1 = make_loss_mesh((theta_hat[0] - 0.9, theta_hat[0] + 0.9), (theta_hat[1] - 0.9, theta_hat[1] + 0.9), points=130)
Z = evaluate_objective_mesh(T0, T1, objective)

fig = plt.figure(figsize=(16, 9))
ax_surface = fig.add_subplot(2, 2, 1, projection="3d")
plot_3d_landscape(ax_surface, T0, T1, Z, "3D training loss surface", path=path, objective_fn=objective, selected=path[best_t])

ax_curves = fig.add_subplot(2, 2, 2)
ax_curves.plot(train_history, color=COLORS["selected"], lw=2.1, label="training MSE")
ax_curves.plot(val_history, color=COLORS["validation"], lw=2.1, label="validation MSE")
for label, t in checkpoints.items():
    ax_curves.axvline(t, ls="--" if label != "validation-selected" else "-", lw=1.3, alpha=0.8, label=f"{label} t={t}")
ax_curves.set_title("Stopping rule selects an iterate")
ax_curves.set_xlabel("update t")
ax_curves.set_ylabel("MSE")
ax_curves.set_yscale("log")
ax_curves.legend(fontsize=7)

ax_contour = fig.add_subplot(2, 2, 3)
levels = np.linspace(float(np.min(Z)), float(np.quantile(Z, 0.92)), 24)
ax_contour.contourf(T0, T1, Z, levels=levels, cmap=VIRIDIS_CMAP)
ax_contour.plot(path[:, 0], path[:, 1], color=COLORS["selected"], lw=2.0, marker="o", markevery=20, ms=3.0)
for label, t in checkpoints.items():
    ax_contour.scatter(path[t, 0], path[t, 1], s=70, label=label)
ax_contour.set_title("Early stopping on a 2D loss landscape")
ax_contour.set_xlabel(r"$\theta_0$")
ax_contour.set_ylabel(r"$\theta_1$")
ax_contour.legend(fontsize=8)

ax_functions = fig.add_subplot(2, 2, 4)
ax_functions.plot(x_val, y_val, color=COLORS["truth"], lw=2.3, label="latent reference")
plot_observed(ax_functions, x_train, y_train, label="training data")
for label, t in checkpoints.items():
    color = COLORS["regularised"] if label == "early" else COLORS["selected"] if label == "validation-selected" else COLORS["alt"]
    lw = 3.0 if label == "validation-selected" else 2.0
    plot_linear_fit(ax_functions, path[t], x_grid=x_val, label=f"{label} h_theta_t", color=color, lw=lw)
ax_functions.set_title("Selected two-parameter functions")
ax_functions.set_xlabel("x")
ax_functions.set_ylabel("y")
ax_functions.set_ylim(-5.4, 3.1)
ax_functions.legend(fontsize=7)
plt.tight_layout()
plt.show()

rows = []
for label, t in checkpoints.items():
    rows.append((label, t, train_history[t], val_history[t], path[t]))

display_table(["Stopping rule", "t", "Training MSE", "Validation MSE", "theta_t"], rows)


> **Researcher note.** A reported model selected by validation stopping is not simply “the trained model.” It is the iterate chosen by a rule, and that rule is part of the evidence behind the research claim.

> **Checkpoint.** Did we change $\mathcal{D}$, $\mathcal{H}$, $\mathcal{O}$, the estimand, or the deployment distribution? The trajectory was fixed; the stopping rule in $\mathcal{O}$ decided which $h_{\theta_t}$ became $s$.


## 10. Exit diagnosis: which part of $\mathcal{O}$ changed?

Optimisation converts candidates into a selected solution:

$$
\mathcal{O}: (\mathcal{H},\mathcal{D})\mapsto s.
$$

In practice, $\mathcal{O}$ includes:

$$
\text{loss} + \text{regularisation} + \text{initialisation} + \text{update rule} + \text{stochasticity} + \text{stopping rule}.
$$

The full workshop frame is now visible:

$$
\boxed{\mathcal{H}} + \boxed{\mathcal{D}} + \boxed{\mathcal{O}} \rightarrow s.
$$

Two learners can share the same data and hypothesis space but select different solutions because their optimisation procedures differ:

$$
(\mathcal{H},\mathcal{D},\mathcal{O}_1)\rightarrow s_1,
\qquad
(\mathcal{H},\mathcal{D},\mathcal{O}_2)\rightarrow s_2.
$$

The next notebook can ask how we investigate a trained solution: what evidence supports it, where it fails, and which part of $\mathcal{H}+\mathcal{D}+\mathcal{O}$ explains the behaviour we observe.


In [ ]:
display_markdown(
    markdown_table(
        ["Observation", "Likely source", "What to inspect"],
        [
            ("Outlier changes fitted line dramatically", "Loss", "Residual penalty and influence."),
            ("Same data and two-parameter H, smoother line selected", "Regularisation", "lambda, penalty form, coefficient norm."),
            ("Objective decreases slowly in one direction", "Conditioning", "Feature scaling and Hessian spectrum."),
            ("Different random seeds give different functions", "Initialisation / stochasticity", "Basin, batch size, seed sensitivity."),
            ("Late model fits noise after validation error rises", "Stopping", "Training and validation curves."),
        ],
    )
)

display_markdown(
    "> **Checkpoint.** For each row, ask: did we change $\\mathcal{D}$, $\\mathcal{H}$, $\\mathcal{O}$, "
    "the estimand, or the deployment distribution? In this notebook, the intended answer is usually a component of $\\mathcal{O}$."
)


### Failure diagnosis exercise

| Observation | Likely source | What to check next |
|---|---|---|
| Good random-test error, bad shifted-context error | Data / robustness | Compare $P_{\mathrm{train}}$ and $P_{\mathrm{deploy}}$. |
| Models disagree in a data gap | Data support + hypothesis bias | Inspect coverage and version-space disagreement. |
| Training loss high for all capacities | Hypothesis or optimisation | Separate approximation failure from training failure. |
| Training loss low, test loss high | Generalisation or distribution mismatch | Inspect validation design, regularisation, and support. |
| Same data and $\mathcal{H}$, different solutions | Optimisation | Inspect initialisation, optimiser, stochasticity, and stopping. |
